In [45]:
import sys
from pathlib import Path

import torch
from torchvision import models

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb

torch_nb.print_setup()

class MobileNetWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        try:
            self.m = models.mobilenet_v3_small(weights=None)
        except TypeError:
            self.m = models.mobilenet_v3_small(pretrained=False)
        self.m.eval()

    def forward(self, x):
        return self.m(x)

model = MobileNetWrapper()


Repository root -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon
torch-mlir-opt  -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/third-party/torch-mlir/build/bin/torch-mlir-opt
Artifacts dir   -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts


In [46]:
import torch
from torch_mlir import fx

mobilenet_module = MobileNetWrapper().eval()
example_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

torch_module = fx.export_and_import(mobilenet_module, example_input, func_name="kernel")
torch_ir = torch_module.operation.get_asm()

torch_file = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_torch.mlir"
torch_file.write_text(torch_ir)

print(f"Wrote Torch dialect IR → {torch_file.resolve()}")
print(torch_ir[:100])

Wrote Torch dialect IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/mobilenet_v3_small_torch.mlir
module {
  func.func @kernel(%arg0: !torch.vtensor<[1,3,224,224],f32>) -> !torch.vtensor<[1,1000],f3


In [31]:
from pathlib import Path

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
linalg_txt = linalg_ir.read_text()
print(linalg_txt[:800])

Wrote Linalg IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/mobilenet_v3_small_linalg.mlir
#map = affine_map<(d0) -> (d0)>
#map1 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map2 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map3 = affine_map<() -> ()>
#map4 = affine_map<(d0, d1, d2, d3) -> ()>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map6 = affine_map<(d0, d1) -> (d0, d1)>
#map7 = affine_map<(d0, d1) -> (d1)>
#map8 = affine_map<(d0, d1) -> ()>
module {
  func.func @kernel(%arg0: tensor<1x3x224x224xf32>) -> tensor<1x1000xf32> {
    %cst = arith.constant dense_resource<torch_tensor_16_3_3_3_torch.float32> : tensor<16x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant 1.000000e+00 : f32
    %cst_2 = arith.constant 1.000000e-03 : f64
    %cst_3 = arith.constant 3.000000e+00 : f32
    %cst_4 = arith.constant 6.000000e+00 : f32


In [47]:
from tutorial._infra import cinm_frontend as cinm_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

mobilenet_linalg_im2col_clean = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_linalg_im2col_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_linalg_im2col_clean,
    ]
)

print(mobilenet_linalg_im2col_clean.read_text()[:2000])

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [33]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

mobilenet_cinm0 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_linalg_im2col_clean,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm0,
    ]
)

print(mobilenet_cinm0.read_text()[:2000])

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [48]:
cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

mobilenet_cinm1 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm0,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm1,
    ]
)

print(mobilenet_cinm1.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [49]:
cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

mobilenet_cinm2 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm1,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm2,
    ]
)

print(mobilenet_cinm2.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [50]:
cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

mobilenet_cinm3 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm2,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm3,
    ]
)

print(mobilenet_cinm3.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [57]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

mobilenet_cinm4 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm3,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm4,
    ]
)

print(mobilenet_cinm4.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [58]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

mobilenet_cinm5 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm4,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm5,
    ]
)

print(mobilenet_cinm5.read_text()[:2000])

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> ((s0 floordiv 112) * 2 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 * 2 + s1 - (s0 floordiv 112) * 224 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0) -> (d0)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map6 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map7 = affine_map<() -> ()>
#map8 = affine_map<(d0, d1, d2, d3) -> ()>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map10 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map11 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map12 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map15 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map16 = affine_map<()[s0] -> (s0 floordiv 56)>
#map17 = affine_map<()[s0] -> (s0 mod 

In [84]:
cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

mobilenet_cinm6 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm5,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm6,
    ]
)

print(mobilenet_cinm6.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> (d0)>
#map2 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map4 = affine_map<() -> ()>
#map5 = affine_map<(d0, d1, d2, d3) -> ()>
#map6 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map7 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map8 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map9 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map10 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map11 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map12 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map13 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 + d4, d3 + d5)>
#map14 = affine_map<(d0, d1) -> (d0, d1)>
#map15 = affine_map<(d0, d1) -> (d1)>
#map16 = affine_map<(d0, d1) -> ()>
module {
  func.func @kernel(%arg0: tensor<1x3x224x224xf32>) -> tensor<1x1000xf32> {
    %c0 = arith.constant 0 : index
 

In [85]:
cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries}"
    ")"
)

mobilenet_cinm7 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm6,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm7,
    ]
)

print(mobilenet_cinm7.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map5 = affine_map<() -> ()>
#map6 = affine_map<(d0, d1, d2, d3) -> ()>
#map7 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d1, d2)>
#map8 = affine_map<(d0, d1, d2) -> (d2, d0, d1)>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d2, d3, d1)>
#map10 = affine_map<(d0, d1, d2) -> (d1, d2, d0)>
#map11 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map12 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map13 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map14 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 + d4, d3 + d5)>
#map15 = affine_map<(d0, d1) -> (d1, d0)>
#map16 = affine_map<(d0, d1) -> (d0, d1)>
#map17 = affine_map<(d0, d1) -> (d1)>
#map18 = affine_map<(d0, d1) -> ()>
module {
  memref.global "private" constant @__

In [89]:
cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add,relu}),"
    "cinm-memory-cleanup"
    ")"
)

mobilenet_cinm8 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm7,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm8,
    ]
)

print(mobilenet_cinm8.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map5 = affine_map<() -> ()>
#map6 = affine_map<(d0, d1, d2, d3) -> ()>
#map7 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map8 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map10 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 + d4, d3 + d5)>
#map11 = affine_map<(d0, d1) -> (d0, d1)>
#map12 = affine_map<(d0, d1) -> (d1)>
#map13 = affine_map<(d0, d1) -> ()>
module {
  memref.global "private" constant @__constant_xi64_1 : memref<i64> = dense<0> {alignment = 64 : i64}
  memref.global "private" constant @__constant_xi64_0 : memref<i64> = dense<1> {alignment = 64 : i64}
  memref.global "private" constant @__constant_xi64 : memref<i64> = dense<6> {alignment = 64

In [97]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported)"
    ")"
)

mobilenet_cinm9 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm8,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm9,
    ]
)

print(mobilenet_cinm9.read_text()[:2000])


#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d1, 0, 0)>
#map5 = affine_map<() -> ()>
#map6 = affine_map<(d0, d1, d2, d3) -> ()>
#map7 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 * 2 + d4, d3 * 2 + d5)>
#map8 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3, d4, d5)>
#map9 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
#map10 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2 + d4, d3 + d5)>
#map11 = affine_map<(d0, d1) -> (d0, d1)>
#map12 = affine_map<(d0, d1) -> (d1)>
#map13 = affine_map<(d0, d1) -> ()>
module {
  memref.global "private" constant @__constant_xi64_1 : memref<i64> = dense<0> {alignment = 64 : i64}
  memref.global "private" constant @__constant_xi64_0 : memref<i64> = dense<1> {alignment = 64 : i64}
  memref.global "private" constant @__constant_xi64 : memref<i64> = dense<6> {alignment = 64

In [98]:
cinm_pipeline = (
    "builtin.module("
    "convert-alpine-to-func,"
    "func.func(convert-linalg-to-loops),"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-scf-to-cf,"
    "expand-strided-metadata,"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-vector-to-llvm,"
    "convert-math-to-llvm,"
    "convert-arith-to-llvm,"
    "convert-index-to-llvm,"
    "convert-to-llvm,"
    "func.func(llvm-request-c-wrappers),"
    "reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

mobilenet_cinm10 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm10.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm10,
    ]
)

print(mobilenet_cinm10.read_text()[:2000])


module {
  llvm.mlir.global private constant @assert_msg_32(dense<[117, 110, 105, 109, 112, 108, 101, 109, 101, 110, 116, 101, 100, 58, 32, 116, 101, 110, 115, 111, 114, 32, 119, 105, 116, 104, 32, 122, 101, 114, 111, 32, 101, 108, 101, 109, 101, 110, 116, 0]> : tensor<40xi8>) {addr_space = 0 : i32} : !llvm.array<40 x i8>
  llvm.mlir.global private constant @assert_msg_31(dense<[117, 110, 105, 109, 112, 108, 101, 109, 101, 110, 116, 101, 100, 58, 32, 116, 101, 110, 115, 111, 114, 32, 119, 105, 116, 104, 32, 122, 101, 114, 111, 32, 101, 108, 101, 109, 101, 110, 116, 0]> : tensor<40xi8>) {addr_space = 0 : i32} : !llvm.array<40 x i8>
  llvm.mlir.global private constant @assert_msg_30(dense<[117, 110, 105, 109, 112, 108, 101, 109, 101, 110, 116, 101, 100, 58, 32, 116, 101, 110, 115, 111, 114, 32, 119, 105, 116, 104, 32, 122, 101, 114, 111, 32, 101, 108, 101, 109, 101, 110, 116, 0]> : tensor<40xi8>) {addr_space = 0 : i32} : !llvm.array<40 x i8>
  llvm.mlir.global private constant @assert_ms

In [100]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb
from tutorial._infra import nbtools

mobilenet_cinm10 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm10.mlir"
mobilenet_cinm10_ll = nbtools.mlir_translate(mobilenet_cinm10)
print(mobilenet_cinm10_ll.read_text()[:2000])

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"

@assert_msg_32 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_31 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_30 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_29 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_28 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_27 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_26 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_25 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_24 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_23 = private constant [40 x i8] c"unimplemented: tensor with zero element\00"
@assert_msg_22 = private con